# Exp10.2 — L1 Membrane Memory and Subthreshold-Evidence Recovery

Aggregation-only notebook for `d1_bb_l1_mem_shift_sweep_v1`. Stage A changes only L1 membrane dynamics with frozen learned weights; Stage B retrains the same D1+BB A2 for each L1 membrane shift.

In [4]:
from pathlib import Path
import json
import pandas as pd

def find_repo_root(start=Path.cwd()):
    for p in (start, *start.parents):
        if (p / 'scripts').exists() and (p / 'notebooks').exists():
            return p
    raise RuntimeError('Repository root not found')

repo = find_repo_root()
root = repo / 'notebooks' / 'artifacts' / 'experiment_10_2_l1_membrane_memory' / 'd1_bb_l1_mem_shift_sweep_v1'
audit = json.loads((root / 'audit.json').read_text())
audit


{'coding': 'bb',
 'dataset': 'D1 postencode_mask only',
 'experiment_id': 'experiment_10_2_l1_membrane_memory',
 'fold_roles': {'0': 'test',
  '1': 'val',
  '2': 'train',
  '3': 'train',
  '4': 'train'},
 'l1_beta': {'1': 0.4999675576146916, '2': 0.75, '3': 0.875, '4': 0.9375},
 'l1_mem_shifts': [1, 2, 3, 4],
 'l1_tau_mem_ms': {'1': 22.54,
  '2': 54.31342963722199,
  '3': 117.0136826471659,
  '4': 242.10347130039656},
 'l2_beta': 0.4999675576146916,
 'l2_mem_shift': 1,
 'l2_tau_mem_ms': 22.54,
 'labels': ['A', 'B', 'C', 'D', 'E', 'G', 'H', 'I', 'J', 'K', 'L', 'X'],
 'model_seeds': [11, 23, 37],
 'objective': 'l2_wcce',
 'paired_randomness': 'For a fixed seed, model initialization and train-loader order are identical across all L1 membrane shifts.',
 'protocol_version': 'd1_bb_l1_mem_shift_sweep_v1',
 'question': 'Can longer L1 membrane memory convert subthreshold analog evidence into binary spike evidence without destroying downstream temporal representation?',
 'readout': 'time_shared

## Stage A — frozen-weight membrane replay

Primary question: does increasing L1 membrane memory make `L1 spike Fixed250 - L1 pre-reset Fixed250` less negative while preserving the pre-reset probe?

In [5]:
stage_a = pd.read_csv(root / 'stage_a_replay_summary.csv')
display(stage_a)

stage_a_contrast = pd.read_csv(root / 'stage_a_replay_shift_contrast_summary.csv')
display(stage_a_contrast)

sanity = pd.read_csv(root / 'stage_a_shift1_replay_sanity.csv')
display(sanity)


,stage,l1_mem_shift,l1_beta,l1_tau_mem_ms,native_train_ba_count,native_train_ba_mean,native_train_ba_std,native_val_ba_count,native_val_ba_mean,native_val_ba_std,...,l2_spike_fixed250_ba_std,l2_quantization_delta_count,l2_quantization_delta_mean,l2_quantization_delta_std,l2_spike_whole_ba_count,l2_spike_whole_ba_mean,l2_spike_whole_ba_std,l2_temporal_ordering_gain_count,l2_temporal_ordering_gain_mean,l2_temporal_ordering_gain_std
0,frozen_replay,1,0.499968,22.540000,3,0.889137,0.017911,3,0.552348,0.016703,...,0.011083,3,-0.028732,0.020664,3,0.539456,0.004039,3,0.052536,0.012829
1,frozen_replay,2,0.750000,54.313430,3,0.606711,0.021578,3,0.432501,0.023085,...,0.010230,3,0.008504,0.015738,3,0.537931,0.023717,3,0.102388,0.014668
2,frozen_replay,3,0.875000,117.013683,3,0.414977,0.022810,3,0.306282,0.042825,...,0.049069,3,0.020106,0.048933,3,0.562893,0.026401,3,0.079619,0.045181
3,frozen_replay,4,0.937500,242.103471,3,0.327479,0.037514,3,0.256114,0.066868,...,0.013021,3,0.064086,0.013259,3,0.546647,0.004594,3,0.123489,0.012013


,stage,l1_mem_shift,native_test_ba_delta_count,native_test_ba_delta_mean,native_test_ba_delta_std,l1_pre_reset_fixed250_ba_delta_count,l1_pre_reset_fixed250_ba_delta_mean,l1_pre_reset_fixed250_ba_delta_std,l1_spike_fixed250_ba_delta_count,l1_spike_fixed250_ba_delta_mean,...,l2_pre_reset_fixed250_ba_delta_std,l2_spike_fixed250_ba_delta_count,l2_spike_fixed250_ba_delta_mean,l2_spike_fixed250_ba_delta_std,l2_spike_whole_ba_delta_count,l2_spike_whole_ba_delta_mean,l2_spike_whole_ba_delta_std,l2_temporal_ordering_gain_delta_count,l2_temporal_ordering_gain_delta_mean,l2_temporal_ordering_gain_delta_std
0,frozen_replay,2,3,-0.143594,0.042482,3,0.024769,0.029766,3,0.034107,...,0.020213,3,0.048327,0.014284,3,-0.001525,0.019679,3,0.049852,0.010075
1,frozen_replay,3,3,-0.252453,0.052712,3,0.031133,0.024700,3,0.055951,...,0.037965,3,0.050519,0.046527,3,0.023436,0.027521,3,0.027083,0.052797
2,frozen_replay,4,3,-0.304838,0.040017,3,0.023065,0.011569,3,0.036236,...,0.025959,3,0.078144,0.014782,3,0.007191,0.007266,3,0.070953,0.015689


,seed,train_shift1_native_test_ba,replay_shift1_native_test_ba,absolute_delta
0,11,0.527525,0.527525,0.0
1,23,0.508387,0.508387,0.0
2,37,0.526790,0.526790,0.0


## Stage B — end-to-end retraining

The table below keeps Stage A and Stage B side-by-side. Seeds are optimization replicates on one locked user split.

In [6]:
runs = pd.read_csv(root / 'all_runs.csv')
summary = pd.read_csv(root / 'all_summary.csv')
display(summary)

contrast = pd.read_csv(root / 'all_shift_contrast_summary.csv')
display(contrast)


,stage,l1_mem_shift,l1_beta,l1_tau_mem_ms,native_train_ba_count,native_train_ba_mean,native_train_ba_std,native_val_ba_count,native_val_ba_mean,native_val_ba_std,...,l2_spike_fixed250_ba_std,l2_quantization_delta_count,l2_quantization_delta_mean,l2_quantization_delta_std,l2_spike_whole_ba_count,l2_spike_whole_ba_mean,l2_spike_whole_ba_std,l2_temporal_ordering_gain_count,l2_temporal_ordering_gain_mean,l2_temporal_ordering_gain_std
0,e2e_retrain,1,0.499968,22.540000,3,0.889137,0.017911,3,0.552348,0.016703,...,0.011083,3,-0.015110,0.044119,3,0.539456,0.004039,3,0.052536,0.012829
1,e2e_retrain,2,0.750000,54.313430,3,0.962260,0.005358,3,0.576873,0.014925,...,0.034088,3,0.031140,0.018865,3,0.597500,0.018774,3,0.036481,0.019792
2,e2e_retrain,3,0.875000,117.013683,3,0.944058,0.030881,3,0.573806,0.037511,...,0.005886,3,0.041558,0.025383,3,0.577981,0.010066,3,0.067451,0.015088
3,e2e_retrain,4,0.937500,242.103471,3,0.945028,0.027597,3,0.566493,0.021218,...,0.030612,3,0.007023,0.040101,3,0.565555,0.009029,3,0.053751,0.022427
4,frozen_replay,1,0.499968,22.540000,3,0.889137,0.017911,3,0.552348,0.016703,...,0.011083,3,-0.028732,0.020664,3,0.539456,0.004039,3,0.052536,0.012829
5,frozen_replay,2,0.750000,54.313430,3,0.606711,0.021578,3,0.432501,0.023085,...,0.010230,3,0.008504,0.015738,3,0.537931,0.023717,3,0.102388,0.014668
6,frozen_replay,3,0.875000,117.013683,3,0.414977,0.022810,3,0.306282,0.042825,...,0.049069,3,0.020106,0.048933,3,0.562893,0.026401,3,0.079619,0.045181
7,frozen_replay,4,0.937500,242.103471,3,0.327479,0.037514,3,0.256114,0.066868,...,0.013021,3,0.064086,0.013259,3,0.546647,0.004594,3,0.123489,0.012013


,stage,l1_mem_shift,native_test_ba_delta_count,native_test_ba_delta_mean,native_test_ba_delta_std,l1_pre_reset_fixed250_ba_delta_count,l1_pre_reset_fixed250_ba_delta_mean,l1_pre_reset_fixed250_ba_delta_std,l1_spike_fixed250_ba_delta_count,l1_spike_fixed250_ba_delta_mean,...,l2_pre_reset_fixed250_ba_delta_std,l2_spike_fixed250_ba_delta_count,l2_spike_fixed250_ba_delta_mean,l2_spike_fixed250_ba_delta_std,l2_spike_whole_ba_delta_count,l2_spike_whole_ba_delta_mean,l2_spike_whole_ba_delta_std,l2_temporal_ordering_gain_delta_count,l2_temporal_ordering_gain_delta_mean,l2_temporal_ordering_gain_delta_std
0,e2e_retrain,2,3,0.036041,0.008299,3,0.040636,0.013729,3,0.048675,...,0.067881,3,0.041988,0.034643,3,0.058044,0.014761,3,-0.016056,0.020412
1,e2e_retrain,3,3,0.024599,0.028562,3,0.047486,0.022601,3,0.046521,...,0.074858,3,0.053439,0.011919,3,0.038524,0.013992,3,0.014914,0.024872
2,e2e_retrain,4,3,-0.001326,0.022504,3,0.026762,0.022151,3,0.036618,...,0.042580,3,0.027314,0.041343,3,0.026099,0.009075,3,0.001215,0.035184
3,frozen_replay,2,3,-0.143594,0.042482,3,0.024769,0.029766,3,0.034107,...,0.020213,3,0.048327,0.014284,3,-0.001525,0.019679,3,0.049852,0.010075
4,frozen_replay,3,3,-0.252453,0.052712,3,0.031133,0.024700,3,0.055951,...,0.037965,3,0.050519,0.046527,3,0.023436,0.027521,3,0.027083,0.052797
5,frozen_replay,4,3,-0.304838,0.040017,3,0.023065,0.011569,3,0.036236,...,0.025959,3,0.078144,0.014782,3,0.007191,0.007266,3,0.070953,0.015689


## Information-path view

In [7]:
cols = [
    'stage','l1_mem_shift','seed','native_test_ba',
    'l1_pre_reset_fixed250_ba','l1_spike_fixed250_ba','l1_quantization_delta',
    'l2_pre_reset_fixed250_ba','l2_spike_fixed250_ba','l2_spike_whole_ba',
    'l2_temporal_ordering_gain'
]
display(runs[cols].sort_values(['stage','l1_mem_shift','seed']))


,stage,l1_mem_shift,seed,native_test_ba,l1_pre_reset_fixed250_ba,l1_spike_fixed250_ba,l1_quantization_delta,l2_pre_reset_fixed250_ba,l2_spike_fixed250_ba,l2_spike_whole_ba,l2_temporal_ordering_gain
0,e2e_retrain,1,11,0.527525,0.707658,0.610823,-0.096834,0.544335,0.580030,0.539106,0.040924
1,e2e_retrain,1,23,0.508387,0.700448,0.636952,-0.063496,0.639159,0.601913,0.535604,0.066308
2,e2e_retrain,1,37,0.526790,0.695439,0.660780,-0.034659,0.637814,0.594036,0.543659,0.050377
3,e2e_retrain,2,11,0.560022,0.732507,0.648027,-0.084481,0.605057,0.615959,0.598048,0.017911
4,e2e_retrain,2,23,0.553911,0.750230,0.667706,-0.082524,0.564450,0.612687,0.578459,0.034229
5,e2e_retrain,2,37,0.556893,0.742715,0.738848,-0.003867,0.639016,0.673297,0.615994,0.057303
6,e2e_retrain,3,11,0.530000,0.729132,0.644591,-0.084541,0.623943,0.642283,0.575623,0.066660
7,e2e_retrain,3,23,0.565231,0.762775,0.684809,-0.077966,0.573131,0.641790,0.589016,0.052773
8,e2e_retrain,3,37,0.541269,0.754095,0.718720,-0.035375,0.614547,0.652222,0.569303,0.082919
9,e2e_retrain,4,11,0.526256,0.745414,0.666082,-0.079332,0.598638,0.647278,0.575549,0.071729


## Firing/dead/saturation diagnostics

Longer membrane memory is useful only if spike decodability improves without driving the population into a high-rate or saturated regime.

In [8]:
activity = pd.read_csv(root / 'all_activity_summary.csv')
display(activity)


,stage,l1_mem_shift,layer,shift,mean_value_per_neuron_step_count,mean_value_per_neuron_step_mean,mean_value_per_neuron_step_std,dead_neuron_fraction_count,dead_neuron_fraction_mean,dead_neuron_fraction_std,high_rate_neuron_fraction_ge_0p5_count,high_rate_neuron_fraction_ge_0p5_mean,high_rate_neuron_fraction_ge_0p5_std,saturated_neuron_fraction_ge_0p95_count,saturated_neuron_fraction_ge_0p95_mean,saturated_neuron_fraction_ge_0p95_std,mean_spikes_per_neuron_s_count,mean_spikes_per_neuron_s_mean,mean_spikes_per_neuron_s_std
0,e2e_retrain,1,l1,2,3,0.005763,0.001076,3,0.124031,0.035524,3,0.000000,0.000000,3,0.0,0.0,3,0.368830,0.068843
1,e2e_retrain,1,l1,3,3,0.028302,0.002926,3,0.116279,0.023256,3,0.000000,0.000000,3,0.0,0.0,3,1.811313,0.187293
2,e2e_retrain,1,l1,4,3,0.129745,0.018897,3,0.087302,0.054986,3,0.063492,0.027493,3,0.0,0.0,3,8.303694,1.209414
3,e2e_retrain,1,l2,2,3,0.335884,0.018907,3,0.000000,0.000000,3,0.007752,0.013427,3,0.0,0.0,3,21.496552,1.210053
4,e2e_retrain,1,l2,3,3,0.365033,0.011677,3,0.000000,0.000000,3,0.023256,0.000000,3,0.0,0.0,3,23.362116,0.747305
5,e2e_retrain,1,l2,4,3,0.389698,0.013545,3,0.000000,0.000000,3,0.126984,0.049563,3,0.0,0.0,3,24.940702,0.866871
6,e2e_retrain,2,l1,2,3,0.012360,0.001258,3,0.046512,0.000000,3,0.000000,0.000000,3,0.0,0.0,3,0.791066,0.080525
7,e2e_retrain,2,l1,3,3,0.039853,0.004277,3,0.038760,0.013427,3,0.000000,0.000000,3,0.0,0.0,3,2.550604,0.273716
8,e2e_retrain,2,l1,4,3,0.124934,0.007140,3,0.007937,0.013746,3,0.063492,0.013746,3,0.0,0.0,3,7.995764,0.456949
9,e2e_retrain,2,l2,2,3,0.375420,0.005178,3,0.000000,0.000000,3,0.038760,0.026854,3,0.0,0.0,3,24.026906,0.331366


## Frozen replay vs retrained dynamics

In [9]:
comparison = pd.read_csv(root / 'e2e_minus_replay.csv')
display(comparison.sort_values(['l1_mem_shift','seed']))


,stage_e2e,l1_mem_shift,l1_beta_e2e,l1_tau_mem_ms_e2e,l2_mem_shift_e2e,seed,native_train_ba_e2e,native_val_ba_e2e,native_test_ba_e2e,lif_test_ba_e2e,...,l2_quantization_delta_replay,l2_spike_whole_ba_replay,l2_temporal_ordering_gain_replay,best_epoch_replay,stopped_epoch_replay,native_test_ba_e2e_minus_replay,l1_pre_reset_fixed250_ba_e2e_minus_replay,l1_spike_fixed250_ba_e2e_minus_replay,l1_quantization_delta_e2e_minus_replay,l2_spike_fixed250_ba_e2e_minus_replay
0,e2e_retrain,1,0.499968,22.540000,1,11,0.905644,0.535063,0.527525,0.467031,...,-0.005172,0.539106,0.040924,96,NaN,0.000000,0.000000,0.000000,0.000000,0.000000
1,e2e_retrain,1,0.499968,22.540000,1,23,0.870093,0.553580,0.508387,0.429692,...,-0.037247,0.535604,0.066308,81,NaN,0.000000,0.000000,0.000000,0.000000,0.000000
2,e2e_retrain,1,0.499968,22.540000,1,37,0.891676,0.568400,0.526790,0.442417,...,-0.043778,0.543659,0.050377,92,NaN,0.000000,0.000000,0.000000,0.000000,0.000000
3,e2e_retrain,2,0.750000,54.313430,1,11,0.959160,0.562432,0.560022,0.466232,...,0.025021,0.535348,0.099654,96,NaN,0.155791,-0.020915,-0.035286,-0.014371,-0.019044
4,e2e_retrain,2,0.750000,54.313430,1,23,0.968448,0.575947,0.553911,0.464697,...,0.006809,0.515611,0.118231,81,NaN,0.160594,0.059078,0.009143,-0.049936,-0.021155
5,e2e_retrain,2,0.750000,54.313430,1,37,0.959173,0.592240,0.556893,0.472628,...,-0.006318,0.562834,0.089280,92,NaN,0.222521,0.009438,0.069847,0.060409,0.021184
6,e2e_retrain,3,0.875000,117.013683,1,11,0.924503,0.551619,0.530000,0.453421,...,-0.024109,0.533623,0.072117,96,NaN,0.233839,0.016966,-0.035797,-0.052763,0.036543
7,e2e_retrain,3,0.875000,117.013683,1,23,0.979660,0.617116,0.565231,0.485760,...,0.011748,0.584908,0.038659,81,NaN,0.270394,0.026739,-0.014370,-0.041109,0.018222
8,e2e_retrain,3,0.875000,117.013683,1,37,0.928013,0.552683,0.541269,0.417055,...,0.072679,0.570147,0.128082,92,NaN,0.326925,0.005353,0.021878,0.016525,-0.046008
9,e2e_retrain,4,0.937500,242.103471,1,11,0.916613,0.576639,0.526256,0.444332,...,0.071209,0.541997,0.129707,96,NaN,0.290850,0.003208,-0.010673,-0.013881,-0.024426


## Interpretation guide

- **Recovery:** L1 spike Fixed250 rises while L1 pre-reset Fixed250 is preserved, so the quantization delta becomes less negative.
- **Fake gap closure:** the gap shrinks only because pre-reset decodability falls.
- **Temporal smearing:** L1 gap may improve but L2 Fixed250/native BA falls as membrane memory becomes too long.
- **Persistent/saturated regime:** high-rate or >=95%-active neuron fractions rise strongly with shift.
- **Training adaptation:** E2E outperforms frozen replay at the same shift, showing weights can reorganize around the longer membrane dynamics.